# 07 — Error analysis

After we have a best model, dig into where it's wrong:
1. Most-confident wrong predictions (per source).
2. Top-weighted tokens per class for the classical model — and how their prevalence shifts year-over-year.
3. Year-stratified accuracy table.

These figures land in `reports/figures/` and feed the report's exploratory section.

In [ ]:
import joblib, pandas as pd, numpy as np
from src.eval.metrics import labels_to_ints, ints_to_labels

pipe = joblib.load('models/classical_v2_random.joblib')
test = pd.read_csv('data/processed/splits_random/test.csv')
X = test['headline'].astype(str).tolist()
y = labels_to_ints(test['source'].tolist())
probs = pipe.predict_proba(X)
preds = probs.argmax(axis=1)
test = test.copy()
test['pred'] = ints_to_labels(preds)
test['confidence'] = probs.max(axis=1)
test['wrong'] = test['pred'] != test['source']
wrong = test[test['wrong']].sort_values('confidence', ascending=False)
print('Most-confident wrong predictions:')
wrong.head(20)[['source', 'pred', 'confidence', 'headline', 'year']]

In [ ]:
# Top-weighted tokens for the classical model
feats = pipe.named_steps['features']
lr = pipe.named_steps['lr']
names = feats.get_feature_names_out()
coefs = lr.coef_[0]
top_fox = sorted(zip(names, coefs), key=lambda x: -x[1])[:30]
top_nbc = sorted(zip(names, coefs), key=lambda x: x[1])[:30]
print('Top-30 toward FoxNews:'); print(top_fox[:15])
print('Top-30 toward NBC:'); print(top_nbc[:15])

In [ ]:
# Year-stratified accuracy — both models, both splits
year_acc = test.groupby('year').apply(lambda g: (g['pred'] == g['source']).mean()).rename('accuracy')
year_n = test.groupby('year').size().rename('n')
pd.concat([year_n, year_acc], axis=1)

In [ ]:
# Same analysis on the temporal test set
test_t = pd.read_csv('data/processed/splits_temporal/test.csv')
X = test_t['headline'].astype(str).tolist()
y = labels_to_ints(test_t['source'].tolist())
probs_t = pipe.predict_proba(X)
preds_t = probs_t.argmax(axis=1)
test_t['pred'] = ints_to_labels(preds_t)
test_t['confidence'] = probs_t.max(axis=1)
test_t['wrong'] = test_t['pred'] != test_t['source']
print('Temporal test acc:', (test_t['pred'] == test_t['source']).mean())
print('Most-confident wrong predictions on 2024+ data:')
test_t[test_t['wrong']].sort_values('confidence', ascending=False).head(20)[['source', 'pred', 'confidence', 'headline', 'year']]